# Beag NIST Compliance Model Training

Train a domain-specific LoRA adapter that maps documents to NIST 800-53 / CSF / CMMC controls.

**Hardware**: Kaggle T4 GPU (16 GB VRAM)
**Base model**: Qwen2.5-7B-Instruct (4-bit QLoRA)
**Training recipe**: CISPO loss + instruction tuning (loss masked to assistant tokens)
**Runtime**: 800 examples = ~90-120 min

## 1. Clone repo

In [ ]:
import os, sys, json, time, subprocess, gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OMP_NUM_THREADS"] = "4"
from pathlib import Path

REPO_URL = "https://github.com/YOUR_ORG/beag-training.git"
REPO_DIR = Path("/kaggle/working/beag-training")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned.")

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Working dir: {os.getcwd()}")

## 2. Set API key (MUST run before project imports)

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["DEEPSEEK_API_KEY"] = UserSecretsClient().get_secret("DEEPSEEK_API_KEY")
    print("API key loaded from Kaggle Secrets.")
except Exception:
    pass

# os.environ["DEEPSEEK_API_KEY"] = "sk-..."

api_key = os.environ.get("DEEPSEEK_API_KEY", "")
if not api_key:
    raise RuntimeError("DEEPSEEK_API_KEY not set. Use Kaggle Secrets or paste it above.")
print(f"API key: {'***' + api_key[-4:] if len(api_key) > 4 else 'SET'}")

## 3. Install dependencies

In [ ]:
try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
except Exception:
    print("WARNING: No GPU detected.")

In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q unsloth
!pip install -q openai pydantic-settings python-dotenv pyyaml scikit-learn pyarrow sentence-transformers

## 4. Import project modules

In [ ]:
from data import DataGenerator, GeneratorConfig
from data.validator import filter_valid
from data.augment import DataAugmenter
from frameworks.catalog import Framework, load_catalog
from frontier.deepseek import DeepSeekClient

print("Project imports OK")

from openai import AsyncOpenAI
test_client = AsyncOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    timeout=15,
)
test_resp = await test_client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "say ok"}],
    max_tokens=10,
)
print(f"DeepSeek API: {test_resp.choices[0].message.content}")

## 5. Generate synthetic data (800 examples)

In [ ]:
NUM_EXAMPLES = 800       # 800 = ~8-10 min
CONCURRENCY = 10
AMBIGUITY_PROB = 0.5

output_dir = REPO_DIR / "output"
output_dir.mkdir(exist_ok=True)

cat = load_catalog()
print(f"Catalog: {len(list(cat.list_by_framework(Framework.NIST_800_53)))} NIST 800-53 controls")

In [ ]:
config = GeneratorConfig(
    total_examples=NUM_EXAMPLES,
    frameworks=list(Framework),
    concurrency=CONCURRENCY,
    max_mappings_per_example=6,
    min_mappings_per_example=3,
    ambiguity_prob=AMBIGUITY_PROB,
)
gen = DataGenerator(
    client=DeepSeekClient(api_key=api_key),
    catalog=cat,
    config=config,
)

print(f"Generating {NUM_EXAMPLES} examples...")
t0 = time.monotonic()
examples = await gen.generate_all_async()
examples, invalid = filter_valid(examples, cat)
print(f"Valid: {len(examples)}  Invalid: {len(invalid)}  Time: {time.monotonic()-t0:.0f}s")

In [ ]:
if not examples:
    raise RuntimeError("0 valid examples generated.")

augmenter = DataAugmenter(cat)
augmented = []
for ex in examples[:max(1, len(examples)//5)]:
    augmented.extend(augmenter.augment(ex))
aug_valid, _ = filter_valid(augmented, cat)
all_examples = examples + aug_valid

train_path = output_dir / "generated_augmented.jsonl"
with open(train_path, "w") as f:
    for ex in all_examples:
        record = {"text": json.dumps(ex.input_json), "mappings": [m.to_dict() for m in ex.mappings], "text_context": ex.text_context}
        f.write(json.dumps(record) + "\n")
print(f"Saved {len(all_examples)} examples to {train_path}")

# Convert to instruction format
from data import records_to_instructions
with open(train_path) as f:
    train_records = [json.loads(l) for l in f.readlines() if l.strip()]
instr_records = records_to_instructions(train_records)
print(f"Converted {len(instr_records)} to instruction pairs")
print(f"  Sample: {instr_records[0]['messages'][-1]['content'][:80]}...")

## 6. Train with QLoRA + CISPO

In [ ]:
import torch
from onprem.unsloth_adapter import UnslothAdapter
from onprem.train import OnPremTrainer

MODEL_TIER = "standard"
LORA_RANK = 32
BATCH_SIZE = 1
EPOCHS = 3
MAX_SEQ_LENGTH = 2048

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

trainer_config = {
    "batch_size": BATCH_SIZE, "num_epochs": EPOCHS,
    "max_seq_length": MAX_SEQ_LENGTH, "learning_rate": 2e-5,
    "gradient_accumulation": 8, "warmup_steps": 50, "max_steps": 500,
    "output_dir": str(output_dir / "checkpoints"),
    "task_type": "nist_multi_label", "catalog": cat,
}

gc.collect(); torch.cuda.empty_cache()

In [ ]:
adapter = UnslothAdapter(
    model_tier=MODEL_TIER, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, lora_rank=LORA_RANK, lora_alpha=16,
)
model, tokenizer = adapter.load()
print(f"Loaded: {adapter.model_id}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")# Verify 4-bit quantization took effect
actual_vram = torch.cuda.memory_allocated()/1e9
if actual_vram > 10:
    print(f"WARNING: Model using {actual_vram:.1f} GB — 4-bit likely failed. Switch to starter tier.")
elif actual_vram > 6:
    print(f"WARNING: Model using {actual_vram:.1f} GB — higher than expected.")
else:
    print(f"4-bit OK: model uses {actual_vram:.1f} GB")


In [ ]:
trainer = OnPremTrainer(model, tokenizer, config=trainer_config)
t0 = time.monotonic()
result = trainer.train(instr_records)
elapsed = time.monotonic() - t0

print(f"\nTraining done ({elapsed:.0f}s)")
print(f"  Steps: {result['total_steps']}")
print(f"  Best loss: {result['best_loss']:.4f}")
print(f"  Final loss: {result['final_loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt
history = result.get("history", {})
if history.get("loss"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.plot(history["loss"]); ax1.set_title("Training Loss")
    ax2.plot(history["lr"]); ax2.set_title("Learning Rate")
    plt.tight_layout()
    plt.savefig(str(output_dir / "training_curves.png"), dpi=100)
    plt.show()

## 6b. OPD Recovery (optional)

Restore instruction-following after domain fine-tune by distilling from the base model over Tulu3 prompts.

In [ ]:
ENABLE_OPD = False
OPD_PROMPTS_PATH = "/kaggle/input/opd-prompts/tulu3-prompts.jsonl"

if ENABLE_OPD and Path(OPD_PROMPTS_PATH).exists():
    from onprem.recipe import OPDRecovery, load_prompts_from_file
    print("Loading teacher model...")
    teacher_adapter = UnslothAdapter(
        model_tier=MODEL_TIER, max_seq_length=2048, load_in_4bit=True,
    )
    teacher_model, _ = teacher_adapter.load()
    teacher_model.eval()
    prompts = load_prompts_from_file(OPD_PROMPTS_PATH)
    print(f"OPD with {len(prompts)} prompts...")
    opd = OPDRecovery(model, teacher_model, tokenizer, {
        "opd_steps": 50, "opd_batch_size": 2, "opd_max_tokens": 128,
        "opd_lr": 5e-6, "promote_every": 10, "temperature": 0.7,
        "max_seq_length": 2048,
    })
    opd_result = opd.recover(prompts)
    print(f"  Steps: {opd_result['opd_steps']}, Final KL: {opd_result['final_loss']:.4f}")
    teacher_adapter.unload()
    del teacher_model, teacher_adapter
    gc.collect(); torch.cuda.empty_cache()
else:
    print("OPD skipped. Set ENABLE_OPD=True with Tulu3 prompts to run.")

## 7. Save model

In [ ]:
adapter_dir = Path("/kaggle/working/beag-nist-adapter")
adapter.save_adapter(str(adapter_dir))

summary = {
    "base_model": adapter.model_id, "tier": MODEL_TIER,
    "lora_rank": LORA_RANK, "examples": len(instr_records),
    "epochs": EPOCHS, "best_loss": result["best_loss"],
    "final_loss": result["final_loss"], "total_steps": result["total_steps"],
}
with open(adapter_dir / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

size_mb = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
print(f"Saved to: {adapter_dir} ({size_mb:.1f} MB)")

## 8. Inference test (with catalog grounding)

In [ ]:
from unsloth import FastLanguageModel
from data import SYSTEM_PROMPT
from frameworks.grounding import ground_control_ids

model_inf, tok_inf = FastLanguageModel.from_pretrained(
    model_name=str(adapter_dir),
    max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(model_inf)

test_doc = json.dumps({
    "type": "policy",
    "title": "Access Control Policy",
    "text": "All users must authenticate via multi-factor authentication before accessing protected systems.",
})

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_doc},
]

inputs = tok_inf.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
).to("cuda")

outputs = model_inf.generate(
    inputs["input_ids"] if hasattr(inputs, "input_ids") else inputs,
    attention_mask=inputs["attention_mask"] if hasattr(inputs, "attention_mask") else None,
    max_new_tokens=512, temperature=0.1, do_sample=True,
)

response = tok_inf.decode(
    outputs[0][inputs["input_ids"].shape[1]:] if hasattr(inputs, "input_ids") else outputs[0][inputs.shape[1]:],
    skip_special_tokens=True,
)

print("Raw model output:")
try:
    parsed = json.loads(response)
    if isinstance(parsed, list):
        grounded, stats = ground_control_ids(parsed, cat)
        print(f"\nGrounding: {stats}")
        print("Grounded mappings (corrected hallucinated IDs):")
        print(json.dumps([{
            "control_id": m["control_id"],
            "framework": m.get("framework", ""),
            "validated": m.get("validated", True),
            "corrected": m.get("corrected", False),
            "confidence": m.get("confidence", 0),
            "reasoning": m.get("reasoning", "")[:100],
        } for m in grounded], indent=2))
    else:
        print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print(response[:500])
    print("...")